In [ ]:
from fastai.conv_learner import *
from fastai.dataset import *
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
import matplotlib.pyplot as plt
import math


In [ ]:
MODEL_NAME = 'Resnet50'
PATH  = "./input"
TRAIN = f'{PATH}/train/'
TEST = f'{PATH}/test/'
LABELS = f'{PATH}/train.csv'
SAMPLE_SUB = f'{PATH}/sample_submission.csv'
BBOX = f'{PATH}/bboxes.csv'

In [ ]:
# Backbone architecture
arch = resnet50
# Number of workers for data preprocessing
num_workers = 8

In [ ]:
df = pd.read_csv(LABELS).set_index('Image')

In [ ]:
new_whale_df = df[df.Id == "new_whale"] # only new_whale dataset
train_df = df[~(df.Id == "new_whale")] # no new_whale dataset, used for training

In [ ]:
unique_labels = np.unique(train_df.Id.values)
labels_dict = dict()
labels_list = []

for i in range(len(unique_labels)):
    labels_dict[unique_labels[i]] = i
    labels_list.append(unique_labels[i])

print("Number of classes: {}".format(len(unique_labels)))
train_names = train_df.index.values
train_df.Id = train_df.Id.apply(lambda x: labels_dict[x])
train_labels = np.asarray(train_df.Id.values)
test_names = [f for f in os.listdir(TEST)]

In [ ]:
labels_count = train_df.Id.value_counts()
dup = []
for idx,row in train_df.iterrows():
    if labels_count[row['Id']] < 5:
        dup.extend([idx]*math.ceil((5 - labels_count[row['Id']])/labels_count[row['Id']]))
train_names = np.concatenate([train_names, dup])
train_names = train_names[np.random.RandomState(seed=42).permutation(train_names.shape[0])]

In [ ]:
plt.figure(figsize=(18, 4))
plt.subplot(121)
_, _,_ = plt.hist(labels_count.values)
plt.ylabel("frequency")
plt.xlabel("class size")

plt.title('class distribution; log scale')
labels_count.head()

print("Count for class new_whale: {}".format(labels_count[0]))
plt.subplot(122)
_ = plt.plot(labels_count[1:].values)
plt.title('w/o class new_whale; log scale')
plt.xlabel("class")
plt.ylabel("log(size)")
plt.gca().set_yscale('log')

In [ ]:
train_df['image_name'] = train_df.index
bbox_df = pd.read_csv(BBOX).set_index('Image')

In [ ]:
train_df.head()

In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42069)

In [ ]:
split = sss.split(train_names, np.zeros(train_names.shape))
train_indexes = []
val_indexes = []
for train_idx, val_idx in sss.split(train_names, np.zeros(train_names.shape)):
    train_indexes = train_idx
    val_indexes = val_idx

In [ ]:
tr_n = [train_names[i] for i in train_indexes]
val_n = [train_names[i] for i in val_indexes]

In [ ]:
print(len(tr_n), len(val_n))

In [ ]:
class HWIDataset(FilesDataset):
    def __init__(self, fnames, path, transform):
        self.train_df = train_df
        super().__init__(fnames, transform, path)

    def get_x(self, i):
        img = open_image(os.path.join(self.path, self.fnames[i]))
        img = cv2.resize(img, (self.sz, self.sz))
        return img

    def get_y(self, i):
        if (self.path == TEST): return 0
        return self.train_df.loc[self.fnames[i]]['Id']

    def get_c(self):
        return len(unique_labels)

In [ ]:
class HWIDataset1(FilesDataset):
    def __init__(self, fnames, path, transform):
        self.train_df = train_df
        super().__init__(fnames, transform, path)

    def get_x(self, i):
        img = open_image(os.path.join(self.path, self.fnames[i]))
        bbox = bbox_df.loc[self.fnames[i]]
        x0, y0, x1, y1 = bbox['x0'], bbox['y0'], bbox['x1'],  bbox['y1']
        if not (x0 >= x1 or y0 >= y1):
            img = img[y0:y1, x0:x1,:]
        img = cv2.resize(img, (self.sz, self.sz))
        return img

    def get_y(self, i):
        if (self.path == TEST): return 0
        return self.train_df.loc[self.fnames[i]]['Id']

    def get_c(self):
        return len(unique_labels)



In [ ]:
def get_data(sz, batch_size):
    """
    Read data and do augmentations
    """
    aug_tfms = [RandomRotateZoom(deg=20, zoom=2, stretch=1),
                RandomLighting(0.2, 0.2),
                RandomCrop(sz),
                RandomBlur(blur_strengths=3),
                RandomFlip()]
    tfms = tfms_from_model(arch, sz, crop_type=CropType.NO, aug_tfms=aug_tfms)
    ds = ImageData.get_ds(HWIDataset1, (tr_n[:-(len(tr_n) % batch_size)], TRAIN),
                          (val_n, TRAIN), tfms, test=(test_names, TEST))
    md = ImageData("./", ds, batch_size, num_workers=num_workers, classes=None)
    return md

In [ ]:
!ls models

In [ ]:
image_size = 224
batch_size = 96
md = get_data(image_size, batch_size)

In [ ]:
extra_fc_layers_size = []
learn = ConvLearner.pretrained(arch, md, ps=0.3) 
learn.opt_fn = optim.Adam

In [ ]:
print('Number of layer groups:', len(learn.get_layer_groups()), '\t(first 2 groups is pretrained backbone)')
print('This is our extra thin on top of the backbone Resnet50 architecture:')
learn.get_layer_groups()[2]

In [ ]:
!ls "models"

learn.load("1_resnet53_cyc_2_orig")

In [ ]:
learn.fit(1e-3, 1, cycle_len=1, cycle_save_name="1_resnet55")

In [ ]:
base_lr = 1e-4 # lr for the backbone
fc_lr = 1e-3 # lr for the classifer

lrs = [base_lr, base_lr, fc_lr]
# Freeze backbone and train the classifier for 2 epochs
learn.fit(lrs=lrs, n_cycle=1, cycle_len=1, cycle_save_name="2_resnet55")


In [ ]:
image_size = 384
batch_size = 32
md = get_data(image_size, batch_size)
learn.set_data(md)

In [ ]:
learn.load("10_resnet55_cyc_1")

In [ ]:
base_lr = 1e-4 # lr for the backbone
fc_lr = 1e-3 # lr for the classifer

lrs = [base_lr, base_lr, fc_lr]
# Freeze backbone and train the classifier for 2 epochs
learn.fit(lrs=lrs, n_cycle=1, cycle_len=1, cycle_save_name="3_resnet55")


In [ ]:
base_lr = 1e-4 # lr for the backbone
fc_lr = 1e-3 # lr for the classifer

In [ ]:
learn.lr_find()

In [ ]:
learn.sched.plot()

In [ ]:
learn.freeze()

In [ ]:
learn.fit(1e-2, 1)

In [ ]:
learn.fit(1e-3, n_cycle=1, cycle_len=1, cycle_save_name="4_resnet55")

In [ ]:
# Unfreeze backbone and continue training for 9 epochs
learn.unfreeze()
learn.freeze_to(-2)
learn.fit(1e-3, n_cycle=1, cycle_len=1, cycle_save_name="5_resnet55")

In [ ]:
learn.unfreeze()
lrs = [1e-4, 1e-4, 1e-3]
learn.fit(lrs, n_cycle=3, cycle_len=1, use_clr=(32, 10), cycle_save_name="6_resnet55")

base_lr = 1e-4 # lr for the backbone
fc_lr = 1e-3 # lr for the classifer

lrs = [base_lr, base_lr, fc_lr]

In [ ]:
learn.fit(lrs, n_cycle=3, cycle_len=1, use_clr=(32, 10), cycle_save_name="7_resnet55")

In [ ]:
lrs = [1e-4, 1e-4, 1e-4]
learn.fit(lrs, n_cycle=2, cycle_len=1, use_clr=(32, 10), cycle_save_name="8_resnet55")

In [ ]:
lrs = [1e-5, 1e-4, 5e-4]
learn.fit(lrs, n_cycle=1, cycle_len=1, use_clr=(32, 10), cycle_save_name="9_resnet55")

In [ ]:
lrs = [1e-5, 5e-5, 5e-4]
learn.fit(lrs, n_cycle=2, cycle_len=1, use_clr=(32, 10), cycle_save_name="10_resnet55")

In [ ]:
lrs = [1e-5, 5e-5, 1e-4]
learn.fit(lrs, n_cycle=3, cycle_len=1, use_clr=(32, 10), cycle_save_name="11_resnet55")

In [ ]:
learn.fit(lrs, n_cycle=3, cycle_len=1, use_clr=(32, 10), cycle_save_name="12_resnet55")

In [ ]:
learn.fit(lrs, n_cycle=2, cycle_len=2, use_clr=(32, 10), cycle_save_name="13_resnet55")

In [ ]:
learn.fit(lrs, n_cycle=2, cycle_len=2, use_clr=(32, 10), cycle_save_name="14_resnet55")

In [ ]:
lr = 5e-5
lrs = [lr/100, lr/10, lr]
learn.fit(lrs, n_cycle=1, cycle_len=1, use_clr=(32, 10), cycle_save_name="15_resnet55")

In [ ]:
lr = 5e-5
lrs = [lr/100, lr/10, lr]
learn.fit(lrs, n_cycle=3, cycle_len=2, use_clr=(32, 10), cycle_save_name="16_resnet55")

In [ ]:
lr = 5e-5
lrs = [lr/100, lr/10, lr]
learn.fit(lrs, n_cycle=3, cycle_len=1, use_clr=(32, 10), cycle_save_name="17_resnet55")

# Unfreeze backbone and continue training for 9 epochs
learn.unfreeze()
learn.fit(lrs, n_cycle=3, cycle_len=2, use_clr=(10, 5), cycle_save_name="1_resnet53")

In [ ]:
best_th = 0.38

In [ ]:
preds_t,y_t = learn.TTA(is_test=True,n_aug=8)
np.save("preds_whale55_2", preds_t)

In [ ]:
preds_sans_0 = preds_t

In [ ]:
preds_t = np.stack(preds_t, axis=-1)
preds_t = np.exp(preds_t)
preds_t = preds_t.mean(axis=-1)
preds_t = np.concatenate([np.zeros((preds_t.shape[0],1))+best_th, preds_t],axis=1)

In [ ]:
preds_t = np.load("preds_whale50_4_box_preds.npy")

In [ ]:
np.save("preds_whale55_1_box", preds_t)

In [ ]:
labels_list

In [ ]:
submission_name = "submission55_6__box"
sample_df = pd.read_csv(SAMPLE_SUB)
sample_list = list(sample_df.Image)
labels_list = ["new_whale"]+labels_list if labels_list[0] != "new_whale" else labels_list
pred_list = [[labels_list[i] for i in p.argsort()[-5:][::-1]] for p in preds_t]
pred_dic = dict((key, value) for (key, value) in zip(learn.data.test_ds.fnames,pred_list))
pred_list_cor = [' '.join(pred_dic[id]) for id in sample_list]
df = pd.DataFrame({'Image':sample_list,'Id': pred_list_cor})
df.to_csv(f'{submission_name}.csv', header=True, index=False)
df.head()

In [ ]:
labels_list

In [ ]:
!kaggle competitions submit -c humpback-whale-identification -f "{submission_name}.csv" -m "resnet50 {submission_name} train no whale, split, bbox, 0.838058 acc"

In [ ]:
sample_df = pd.read_csv(SAMPLE_SUB)
sample_list = list(sample_df.Image)
labels_list = ["new_whale"]+labels_list if labels_list[0] != "new_whale" else labels_list
pred_list = [[labels_list[i] for i in p.argsort()[-5:][::-1]] for p in preds_new]
pred_dic = dict((key, value) for (key, value) in zip(md.test_ds.fnames,pred_list))
pred_list_cor = [' '.join(pred_dic[id]) for id in sample_list]
df = pd.DataFrame({'Image':sample_list,'Id': pred_list_cor})
df.to_csv('submission50_ensemble_4_box.csv', header=True, index=False)
df.head()

In [ ]:
labels_list

In [ ]:
!ls

In [ ]:
!kaggle competitions submit -c humpback-whale-identification -f "submission50_4__box.csv" -m "resnet50 v5 train no whale, bbox 4 fix"